In [ ]:
"""
Bidirectional Long Short-Term Memory (BiLSTM) is an extension of LSTM that processes sequences in both forward and backward directions,
allowing the model to capture both past and future context.

Processes sequences in forward and backward directions
Captures both past and future contextual information
More effective than standard LSTMs for sequence understanding
Commonly used in NLP, speech processing and sequence analysis
"""

In [2]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

In [5]:
# Loading and Preparing the IMDB Dataset

# if steel have not downloaded dataset uncomment and use this
# dataset = tfds.load('imdb_reviews', as_supervised=True)
# train_dataset, test_dataset = dataset['train'], dataset['test']

# batch_size = 32

# train_dataset = train_dataset.shuffle(10000).batch(batch_size)
# test_dataset = test_dataset.batch(batch_size)


dataset = tfds.load(
    'imdb_reviews',
    as_supervised=True,
    download=True
)

train_dataset, test_dataset = dataset['train'], dataset['test']

batch_size = 32
train_dataset = train_dataset.shuffle(10000).batch(batch_size)
test_dataset = test_dataset.batch(batch_size)

In [6]:
example, label = next(iter(train_dataset))
print('Text:\n', example.numpy()[0])
print('\nLabel: ', label.numpy()[0])

Text:
 b'Man, I really wanted to like these shows. I am starving for some good television and I applaud TNT for providing these "opportunites". But, sadly, I am in the minority I guess when it comes to the Cinematic Stephen King. As brilliant as King\'s writing is, the irony is that it simply doesn\'t translate well to the screen, big or small. With few exceptions (very few), the King experience cannot be filmed with the same impact that the stories have when read. Many people would disagree with this, but I\'m sure that in their heart of hearts they have to admit that the best filmed King story is but a pale memory of the one they read. The reason is simple. The average King story takes place in the mind-scape of the characters in the story. He gives us glimpses of their inner thoughts, their emotions and their sometimes fractured or unreal points of view. In short, King takes the reader places where you can\'t put a Panavision camera. As an audience watching the filmed King, we\'re l

I0000 00:00:1786972830.135942  101662 tf_record_dataset_op.cc:396] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608
W0000 00:00:1786972830.295968   90333 cache_dataset_ops.cc:912] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


In [7]:
#  Performing Text Vectorization

vectorize_layer = tf.keras.layers.TextVectorization(
    output_mode='int', output_sequence_length=100)

vectorize_layer.adapt(train_dataset.map(lambda x, y: x))

In [8]:
#  Defining Model Architecture (BiLSTM Layers)

model = tf.keras.Sequential([
    vectorize_layer,
    tf.keras.layers.Embedding(
        len(vectorize_layer.get_vocabulary()), 64, mask_zero=True),
    tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(64, return_sequences=True)),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1)
])

model.build(input_shape=(None,))


model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(),
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (None, 100)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 100, 64)        │     7,801,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 100, 128)       │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,912,705 (30.18 MB)

 Trainable params: 7,912,705 (30.18 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
# Training the Model

history = model.fit(
    train_dataset,
    epochs=3,
    validation_data=test_dataset,
)

Epoch 1/3


/home/soheil/DeepLearning /myvenv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
E0000 00:00:1786972890.071070   90333 util.cc:131] oneDNN supports DT_BOOL only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


782/782 ━━━━━━━━━━━━━━━━━━━━ 169s 210ms/step - accuracy: 0.7836 - loss: 0.4358 - val_accuracy: 0.7990 - val_loss: 0.4111
Epoch 2/3
782/782 ━━━━━━━━━━━━━━━━━━━━ 166s 213ms/step - accuracy: 0.9228 - loss: 0.1982 - val_accuracy: 0.7936 - val_loss: 0.5246
Epoch 3/3
782/782 ━━━━━━━━━━━━━━━━━━━━ 166s 212ms/step - accuracy: 0.9730 - loss: 0.0740 - val_accuracy: 0.7697 - val_loss: 0.6992


In [10]:
# Prediction

review = tf.constant(["This movie was amazing and engaging"])
prob = tf.sigmoid(model.predict(review))[0][0]

sentiment = "Positive" if prob >= 0.5 else "Negative"
print(f"Sentiment: {sentiment}, Probability: {prob:.2f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 450ms/step
Sentiment: Positive, Probability: 0.91
